# Dataset Processing

This notebook prepares the canonical training dataset from the raw CSV files in `datasets/`. It validates the source schema, merges the files, removes blank rows and exact duplicates, and saves both the raw merge and the cleaned combined CSV.


## Environment Setup

Install the notebook dependencies and import the libraries used for dataset preparation.


In [ ]:
# Install only the extra packages needed in Colab.
# Leave Colab's torch / torchvision versions alone.
# Reinstalling one without the other can break CUDA support.
%pip install -q transformers datasets accelerate

In [ ]:
from pathlib import Path

import pandas as pd


## Project Configuration

Define the dataset paths, expected schema, and output files for the cleaned dataset artifact.


In [ ]:
# Project paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "datasets"

RAW_COMBINED_PATH = DATA_DIR / "combined_dataset_raw.csv"
COMBINED_INPUT_PATH = DATA_DIR / "combined_dataset.csv"

EXPECTED_COLUMNS = ["text", "label", "persona", "source_type", "scenario"]
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"
EXPECTED_SOURCE_FILE_COUNT = 6


## Dataset Assembly

Find the raw CSV sources, validate their columns, merge them, and save both the raw merged file and the cleaned training dataset.


In [ ]:
# Find the source CSV files
csv_paths = sorted(DATA_DIR.glob("*.csv"))
source_paths = [
    path
    for path in csv_paths
    if path.name not in {RAW_COMBINED_PATH.name, COMBINED_INPUT_PATH.name}
]

# Fail fast if any source files are missing
if len(source_paths) != EXPECTED_SOURCE_FILE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_SOURCE_FILE_COUNT} source CSV files, found {len(source_paths)}: "
        f"{[path.name for path in source_paths]}"
    )

In [ ]:
frames = []
for path in source_paths:
    frame = pd.read_csv(path)

    missing_columns = [col for col in EXPECTED_COLUMNS if col not in frame.columns]
    extra_columns = [col for col in frame.columns if col not in EXPECTED_COLUMNS]

    # Stop if a source file is missing required columns
    if missing_columns:
        raise ValueError(f"{path.name} is missing required columns: {missing_columns}")

    # Warn about columns that will be dropped
    if extra_columns:
        print(f"Warning: {path.name} has extra columns that will be dropped: {extra_columns}")

    # Keep only the columns used downstream
    frame = frame[EXPECTED_COLUMNS].copy()
    # Track which file each row came from
    frame["source_file"] = path.name
    frames.append(frame)

# Save the raw merged dataset
combined_raw_df = pd.concat(frames, ignore_index=True)
combined_raw_df.to_csv(RAW_COMBINED_PATH, index=False)

In [ ]:
# Normalize text and label columns
combined_df = combined_raw_df.copy()
combined_df[TEXT_COLUMN] = combined_df[TEXT_COLUMN].astype(str).str.strip()
combined_df[LABEL_COLUMN] = combined_df[LABEL_COLUMN].astype(str).str.strip()

# Drop rows missing text or label
combined_df = combined_df.loc[
    combined_df[TEXT_COLUMN].ne("") & combined_df[LABEL_COLUMN].ne("")
]

# Drop exact duplicate examples
combined_df = combined_df.drop_duplicates(subset=EXPECTED_COLUMNS).reset_index(drop=True)

# Save the cleaned dataset
combined_df.to_csv(COMBINED_INPUT_PATH, index=False)

merge_summary = pd.DataFrame(
    {
        "file": [path.name for path in source_paths],
        "rows": [len(frame) for frame in frames],
    }
)

display(merge_summary)
print(f"Raw merged rows: {len(combined_raw_df):,}")
print(f"Clean merged rows: {len(combined_df):,}")
print(f"Saved raw merge to: {RAW_COMBINED_PATH}")
print(f"Saved clean merge to: {COMBINED_INPUT_PATH}")

display(combined_df.head())
display(combined_df[LABEL_COLUMN].value_counts().rename_axis("label").to_frame("count"))
